In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

import seaborn as sns
import prettypyplot as pplt
pplt.use_style(colors='tab10', latex=False, ipython=False,
               mode='poster')
import MDAnalysis as mda
import json
import pickle
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D

import prettypyplot as pplt
pplt.use_style(colors='pastel5', latex=False, ipython=False,
               mode='poster')
from sklearn.linear_model import LinearRegression

pplt.update_style(style='default', mode='print', colors='pastel5')

from batter.api import list_fe_runs, load_fe_run

In [ ]:
from __future__ import annotations

from typing import Literal

import numpy as np
import pandas as pd
from cinnabar import plotting 

def _combine_estimates(
    values,
    ses,
    uncertainty_mode: Literal["ivw", "sample", "max"] = "max",
) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    ses = np.asarray(ses, dtype=float)

    if len(values) == 0:
        raise ValueError("No values to combine.")
    if np.any(~np.isfinite(values)):
        raise ValueError("Non-finite values found.")
    if np.any(~np.isfinite(ses)) or np.any(ses <= 0):
        raise ValueError("All uncertainties must be finite and > 0.")

    if len(values) == 1:
        return float(values[0]), float(ses[0])

    weights = 1.0 / np.square(ses)
    mean = float(np.sum(weights * values) / np.sum(weights))
    ivw_se = float(np.sqrt(1.0 / np.sum(weights)))
    sample_se = float(np.std(values, ddof=1) / np.sqrt(len(values)))

    if uncertainty_mode == "ivw":
        out_se = ivw_se
    elif uncertainty_mode == "sample":
        out_se = sample_se
    elif uncertainty_mode == "max":
        out_se = max(ivw_se, sample_se)
    else:
        raise ValueError("uncertainty_mode must be 'ivw', 'sample', or 'max'.")

    return mean, out_se


def _normalize_energy_unit(unit_obj, unit_module):
    if unit_obj is None:
        return unit_module.kilocalorie_per_mole
    if hasattr(unit_obj, "dimensionality"):
        return unit_obj

    text = str(unit_obj).strip().lower()
    mapping = {
        "kcal/mol": unit_module.kilocalorie_per_mole,
        "kilocalorie_per_mole": unit_module.kilocalorie_per_mole,
        "kilocalories_per_mole": unit_module.kilocalorie_per_mole,
        "kj/mol": unit_module.kilojoule_per_mole,
        "kilojoule_per_mole": unit_module.kilojoule_per_mole,
        "kilojoules_per_mole": unit_module.kilojoule_per_mole,
    }
    if text not in mapping:
        raise ValueError(f"Unsupported unit: {unit_obj!r}")
    return mapping[text]


def dataframe_to_cinnabar(
    rbfe_df: pd.DataFrame,
    *,
    ligand_column: str = "ligand",
    dg_column: str = "total_dG",
    se_column: str = "total_se",
    run_column: str = "run_id",
    status_column: str = "status",
    success_value: str = "success",
    temperature_column: str = "temperature",
    edge_separator: str = "~",
    source: str = "RBFE",
    uncertainty_mode: Literal["ivw", "sample", "max"] = "max",
    combine_by_run_first: bool = True,
    experimental_df: pd.DataFrame | None = None,
    exp_ligand_column: str = "ligand",
    exp_abfe_column: str = "abfe",
    exp_error_column: str | None = None,   # <- now optional
    exp_status_column: str | None = None,
    exp_success_value: str = "success",
    exp_temperature_column: str | None = None,
    exp_source: str = "experiment",
    exp_value_unit="kcal/mol",
    exp_error_unit=None,
):
    from cinnabar.femap import FEMap
    from openff.units import unit

    exp_error_unit = exp_value_unit if exp_error_unit is None else exp_error_unit
    exp_value_unit = _normalize_energy_unit(exp_value_unit, unit)
    exp_error_unit = _normalize_energy_unit(exp_error_unit, unit)

    # -------------------------
    # Parse RBFE dataframe
    # -------------------------
    required = {ligand_column, dg_column, se_column}
    missing = required - set(rbfe_df.columns)
    if missing:
        raise ValueError(f"Missing RBFE columns: {sorted(missing)}")

    work = rbfe_df.copy()

    if status_column in work.columns:
        work = work.loc[work[status_column] == success_value].copy()

    work = work.dropna(subset=[ligand_column, dg_column, se_column]).copy()
    if work.empty:
        raise ValueError("No usable RBFE rows remain after filtering.")

    lig_split = work[ligand_column].astype(str).str.split(edge_separator, n=1, expand=True)
    if lig_split.shape[1] != 2:
        raise ValueError(
            f"Could not split '{ligand_column}' using separator '{edge_separator}'."
        )

    work["ligand_A_raw"] = lig_split[0].str.strip()
    work["ligand_B_raw"] = lig_split[1].str.strip()

    forward_is_canonical = work["ligand_A_raw"] <= work["ligand_B_raw"]
    work["labelA"] = np.where(forward_is_canonical, work["ligand_A_raw"], work["ligand_B_raw"])
    work["labelB"] = np.where(forward_is_canonical, work["ligand_B_raw"], work["ligand_A_raw"])

    raw_dg = pd.to_numeric(work[dg_column], errors="raise").astype(float)
    work["signed_dDG"] = np.where(forward_is_canonical, raw_dg, -raw_dg)
    work["input_se"] = pd.to_numeric(work[se_column], errors="raise").astype(float)

    if np.any(work["input_se"] <= 0):
        raise ValueError(f"Column '{se_column}' must contain only positive values.")

    if temperature_column in work.columns:
        work["temperature_K"] = pd.to_numeric(work[temperature_column], errors="coerce")
    else:
        work["temperature_K"] = 298.15

    raw_signed = work.copy()

    def summarize_rbfe_block(g: pd.DataFrame) -> dict:
        mean, out_se = _combine_estimates(
            g["signed_dDG"].values,
            g["input_se"].values,
            uncertainty_mode=uncertainty_mode,
        )
        return {
            "calc_DDG": mean,
            "calc_dDDG": out_se,
            "n_measurements": int(len(g)),
            "temperature_K": float(g["temperature_K"].dropna().mean())
            if g["temperature_K"].notna().any()
            else 298.15,
        }

    if combine_by_run_first and run_column in raw_signed.columns:
        per_run_records = []
        for (labelA, labelB, run_id), g in raw_signed.groupby(
            ["labelA", "labelB", run_column], sort=True
        ):
            rec = {"labelA": labelA, "labelB": labelB, run_column: run_id}
            rec.update(summarize_rbfe_block(g))
            per_run_records.append(rec)
        per_run = pd.DataFrame(per_run_records)

        edge_records = []
        for (labelA, labelB), g in per_run.groupby(["labelA", "labelB"], sort=True):
            mean, out_se = _combine_estimates(
                g["calc_DDG"].values,
                g["calc_dDDG"].values,
                uncertainty_mode=uncertainty_mode,
            )
            edge_records.append(
                {
                    "labelA": labelA,
                    "labelB": labelB,
                    "calc_DDG": mean,
                    "calc_dDDG": out_se,
                    "n_runs": int(len(g)),
                    "n_measurements": int(g["n_measurements"].sum()),
                    "temperature_K": float(g["temperature_K"].mean()),
                }
            )
        edge_summary = pd.DataFrame(edge_records)
    else:
        edge_records = []
        for (labelA, labelB), g in raw_signed.groupby(["labelA", "labelB"], sort=True):
            rec = {"labelA": labelA, "labelB": labelB}
            rec.update(summarize_rbfe_block(g))
            rec["n_runs"] = int(g[run_column].nunique()) if run_column in g.columns else 1
            edge_records.append(rec)
        edge_summary = pd.DataFrame(edge_records)

    femap = FEMap()
    for row in edge_summary.itertuples(index=False):
        femap.add_relative_calculation(
            labelA=row.labelA,
            labelB=row.labelB,
            value=float(row.calc_DDG) * unit.kilocalorie_per_mole,
            uncertainty=float(row.calc_dDDG) * unit.kilocalorie_per_mole,
            source=source,
            temperature=float(row.temperature_K) * unit.kelvin,
        )

    # -------------------------
    # Parse experimental dataframe
    # -------------------------
    exp_summary = None
    if experimental_df is not None:
        exp_required = {exp_ligand_column, exp_abfe_column}
        exp_missing = exp_required - set(experimental_df.columns)
        if exp_missing:
            raise ValueError(f"Missing experimental columns: {sorted(exp_missing)}")

        exp_work = experimental_df.copy()

        if exp_status_column is not None and exp_status_column in exp_work.columns:
            exp_work = exp_work.loc[exp_work[exp_status_column] == exp_success_value].copy()

        drop_cols = [exp_ligand_column, exp_abfe_column]
        if exp_error_column is not None and exp_error_column in exp_work.columns:
            has_exp_error = True
            drop_cols.append(exp_error_column)
        else:
            has_exp_error = False

        exp_work = exp_work.dropna(subset=drop_cols).copy()

        if not exp_work.empty:
            exp_work["label"] = exp_work[exp_ligand_column].astype(str).str.strip()
            exp_work["exp_DG"] = pd.to_numeric(
                exp_work[exp_abfe_column], errors="raise"
            ).astype(float)

            if has_exp_error:
                exp_work["exp_uncertainty"] = pd.to_numeric(
                    exp_work[exp_error_column], errors="raise"
                ).astype(float)
                if np.any(exp_work["exp_uncertainty"] <= 0):
                    raise ValueError(
                        f"Experimental column '{exp_error_column}' must contain only positive values."
                    )
            else:
                exp_work["exp_uncertainty"] = np.nan

            if exp_temperature_column is not None and exp_temperature_column in exp_work.columns:
                exp_work["temperature_K"] = pd.to_numeric(
                    exp_work[exp_temperature_column], errors="coerce"
                )
            else:
                exp_work["temperature_K"] = 298.15

            exp_records = []
            for label, g in exp_work.groupby("label", sort=True):
                if has_exp_error:
                    mean, out_se = _combine_estimates(
                        g["exp_DG"].values,
                        g["exp_uncertainty"].values,
                        uncertainty_mode=uncertainty_mode,
                    )
                else:
                    mean = float(g["exp_DG"].mean())
                    out_se = np.nan

                exp_records.append(
                    {
                        "label": label,
                        "exp_DG": mean,
                        "exp_uncertainty": out_se,
                        "n_exp": int(len(g)),
                        "temperature_K": float(g["temperature_K"].dropna().mean())
                        if g["temperature_K"].notna().any()
                        else 298.15,
                    }
                )

            exp_summary = pd.DataFrame(exp_records)

            for row in exp_summary.itertuples(index=False):
                kwargs = dict(
                    label=row.label,
                    value=float(row.exp_DG) * exp_value_unit,
                    source=exp_source,
                    temperature=float(row.temperature_K) * unit.kelvin,
                )
                if pd.notna(row.exp_uncertainty):
                    kwargs["uncertainty"] = float(row.exp_uncertainty) * exp_error_unit
                else:
                    kwargs["uncertainty"] = 0 * exp_error_unit

                femap.add_experimental_measurement(**kwargs)

    return femap, edge_summary, raw_signed, exp_summary

In [ ]:
base_folder = '/path/to/simulation'
exp_csv = 'reference/exp.csv'
exp_df = pd.read_csv(exp_csv)

In [ ]:
runs_all = list_fe_runs(f'{base_folder}/')

In [ ]:
runs_all.head()

In [ ]:
femap, edge_summary, raw_signed, exp_summary = dataframe_to_cinnabar(
    rbfe_df=runs_all,
    ligand_column="ligand",
    dg_column="total_dG",
    se_column="total_se",
    run_column="run_id",
    experimental_df=exp_df,
    exp_ligand_column="ligand_id",
    exp_abfe_column="exp_dg",
    exp_source="assay",
    exp_value_unit="kcal/mol",
)


In [ ]:
plotting.plot_DDGs(
    femap.to_legacy_graph(), target_name="Example protein", title="Calculated vs Experimental ΔΔG", figsize=5
)